# Phase 3 — Render-only

This notebook **renders an existing `output/review/` dossier**. It does
not run the planner or prebuild. Use it for fast iteration on grade,
backplate, cover-pick, and other render-time knobs after you have
already produced a complete review folder.

Inputs required (all under `/content/output/` after this notebook runs):

| Path                                  | Contents                                          |
| ------------------------------------- | ------------------------------------------------- |
| `output/al_askari_plan_v2.json`       | Shot plan (from Phase 3 planner)                  |
| `output/al_askari_audio.mp3`          | Narration audio (from Phase 2)                    |
| `output/review/decisions.json`        | Dossier with chosen candidates per shot           |
| `output/review/shot_*/`               | Per-shot candidate folders                        |
| `output/review/overrides/`            | Optional user overrides (cover, portrait)         |

Set `SOURCE` in cell 1 to either `"zip"` (upload a packed archive) or
`"drive"` (mount Google Drive and copy from a path you specify).


In [1]:
# Used to securely store your API key
from google.colab import userdata
import os

# Set your Hugging Face token from Colab's secrets manager
hf_token = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = hf_token

print("HF_TOKEN environment variable set.")
print("You may need to restart your runtime for this to take full effect in all processes.")

HF_TOKEN environment variable set.
You may need to restart your runtime for this to take full effect in all processes.


In [2]:
# ════════════════════════════════════════════════════════════════════
# Settings — edit before running. Everything below this cell auto-runs
# from these values.
# ════════════════════════════════════════════════════════════════════

# Where do plan + audio + review/ live?
#   "zip"   — Colab will prompt you to upload one .zip containing all three
#             at the right relative paths (al_askari_plan_v2.json,
#             al_askari_audio.mp3, review/...).
#   "drive" — mount Google Drive and copy from DRIVE_SOURCE_DIR below.
SOURCE = "zip"
# SOURCE = "drive"

# Only used when SOURCE == "drive". Set to the folder in your Drive that
# contains the three inputs at the relative paths above. Trailing slash
# does not matter.
DRIVE_SOURCE_DIR = "/content/drive/MyDrive/_Phase3/sources"

# ── Render knobs (passed directly to render_plan.py) ────────────────
PLAN_FILE      = "output/al_askari_plan_v2.json"
AUDIO_FILE     = "output/al_askari_audio.mp3"
REVIEW_DIR     = "output/review/"
OUTPUT_FILE    = "output/final_cut_B.mp4"

BOOK_COVER_PICK   = 1          # 1-indexed. 1..N where N = files in resources/book_cover/
BOOK_COVER_FIT    = "contain"  # fill | contain | blur_pad
BOOK_COVER_ALIGN  = "left"    # center | left | right
TYPOGRAPHY_FAMILY = "B"        # A | B | C
GRADE             = "warm"     # warm | cool | neutral | bw
CAPTION_BACKPLATE = "off"   # off | subtle | solid

print(f"SOURCE            = {SOURCE!r}")
if SOURCE == "drive":
    print(f"DRIVE_SOURCE_DIR  = {DRIVE_SOURCE_DIR!r}")
print(f"OUTPUT_FILE       = {OUTPUT_FILE!r}")
print(f"GRADE             = {GRADE!r}")
print(f"CAPTION_BACKPLATE = {CAPTION_BACKPLATE!r}")
print(f"TYPOGRAPHY_FAMILY = {TYPOGRAPHY_FAMILY!r}")
print(f"BOOK_COVER_PICK   = {BOOK_COVER_PICK}")


SOURCE            = 'zip'
OUTPUT_FILE       = 'output/final_cut_B.mp4'
GRADE             = 'warm'
CAPTION_BACKPLATE = 'off'
TYPOGRAPHY_FAMILY = 'B'
BOOK_COVER_PICK   = 1


In [3]:
# Script source: GitHub repo
import os
import shutil

github_url_path = "https://github.com/abdoljh/Lamahat/tree/main/_Phase3"

# Extract the base GitHub repository URL and the subfolder path
def parse_github_path(url):
    parts = url.split('/tree/main/')
    repo_url = parts[0]
    subfolder_path = parts[1] if len(parts) > 1 else ''
    # Add .git for cloning
    repo_url_for_clone = repo_url + '.git'
    return repo_url_for_clone, subfolder_path

repo_url_for_clone, subfolder_path_in_repo = parse_github_path(github_url_path)

repo_name = repo_url_for_clone.split('/')[-1].replace('.git', '')
temp_clone_dir = os.path.join('/tmp', repo_name)
dest_dir_colab = '/content'

print(f"Cloning repository: {repo_url_for_clone}")
print(f"Target subfolder in repo: {subfolder_path_in_repo}")

# Clean up any previous clone to avoid issues
if os.path.exists(temp_clone_dir):
    shutil.rmtree(temp_clone_dir)
    print(f"Removed existing temporary directory: {temp_clone_dir}")

# Clone the repository
clone_command = f"git clone {repo_url_for_clone} {temp_clone_dir}"
print(f"Executing: {clone_command}")
os.system(clone_command)

# Check if cloning was successful
if not os.path.exists(temp_clone_dir):
    print(f"Error: Failed to clone repository {repo_url_for_clone}")
else:
    source_dir_to_copy = os.path.join(temp_clone_dir, subfolder_path_in_repo)
    if not os.path.exists(source_dir_to_copy):
        print(f"Error: Subfolder '{subfolder_path_in_repo}' not found in cloned repository at '{source_dir_to_copy}'")
    else:
        print(f"Source directory to copy: {source_dir_to_copy}")
        print(f"Copying contents of '{source_dir_to_copy}' directly into '{dest_dir_colab}'.")

        # Define directories to skip
        dirs_to_skip = ['artifacts', 'review']

        try:
            for item in os.listdir(source_dir_to_copy):
                if item in dirs_to_skip:
                    print(f"Skipping directory '{item}' as requested.")
                    continue

                source_item_path = os.path.join(source_dir_to_copy, item)
                dest_item_path = os.path.join(dest_dir_colab, item)

                # If item already exists in destination, remove it to avoid errors
                if os.path.exists(dest_item_path):
                    if os.path.isdir(dest_item_path):
                        shutil.rmtree(dest_item_path)
                        print(f"Removed existing directory '{dest_item_path}'.")
                    else:
                        os.remove(dest_item_path)
                        print(f"Removed existing file '{dest_item_path}'.")

                if os.path.isdir(source_item_path):
                    shutil.copytree(source_item_path, dest_item_path)
                else:
                    shutil.copy2(source_item_path, dest_item_path)
                print(f"Copied '{source_item_path}' to '{dest_item_path}'.")
            print(f"Successfully copied contents of '{source_dir_to_copy}' to '{dest_dir_colab}'.")

        except Exception as e:
            print(f"An error occurred during directory contents copy: {e}")

# Clean up the cloned repository
if os.path.exists(temp_clone_dir):
    shutil.rmtree(temp_clone_dir)
    print(f"Cleaned up temporary clone directory: {temp_clone_dir}")

print("😺 GitHub repo loaded!")

Cloning repository: https://github.com/abdoljh/Lamahat.git
Target subfolder in repo: _Phase3
Executing: git clone https://github.com/abdoljh/Lamahat.git /tmp/Lamahat
Source directory to copy: /tmp/Lamahat/_Phase3
Copying contents of '/tmp/Lamahat/_Phase3' directly into '/content'.
Copied '/tmp/Lamahat/_Phase3/CLAUDE.md' to '/content/CLAUDE.md'.
Copied '/tmp/Lamahat/_Phase3/sandbox_test.py' to '/content/sandbox_test.py'.
Copied '/tmp/Lamahat/_Phase3/verify_user_marked.py' to '/content/verify_user_marked.py'.
Copied '/tmp/Lamahat/_Phase3/prebuild_assets.py' to '/content/prebuild_assets.py'.
Copied '/tmp/Lamahat/_Phase3/render_plan.py' to '/content/render_plan.py'.
Copied '/tmp/Lamahat/_Phase3/resources' to '/content/resources'.
Copied '/tmp/Lamahat/_Phase3/_phase3_b4b.ipynb' to '/content/_phase3_b4b.ipynb'.
Copied '/tmp/Lamahat/_Phase3/trim_book_cover.py' to '/content/trim_book_cover.py'.
Copied '/tmp/Lamahat/_Phase3/fonts' to '/content/fonts'.
Skipping directory 'review' as requested.
C

In [4]:
# Render-time dependencies. The planner/prebuild are NOT used here, so
# anthropic, pexels, and Whisper are not required. We only need the
# rendering stack: Pillow + Arabic shaping + bidi. Phase3's render.py
# also relies on ffmpeg, which Colab ships preinstalled.
!pip install --quiet pillow arabic-reshaper python-bidi
print("✓ render-time dependencies installed")

# Confirm ffmpeg is on PATH (Colab default — sanity check only)
!ffmpeg -version 2>&1 | head -1


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 299.6/299.6 kB 11.0 MB/s eta 0:00:00
✓ render-time dependencies installed
ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers


In [5]:
# ════════════════════════════════════════════════════════════════════
# Materialise plan + audio + review/ under /content/output/
# ════════════════════════════════════════════════════════════════════
import os
import shutil
import subprocess
from pathlib import Path

OUT = Path("/content/output")
OUT.mkdir(parents=True, exist_ok=True)

def _confirm_inputs():
    plan = OUT / Path(PLAN_FILE).name
    audio = OUT / Path(AUDIO_FILE).name
    review = OUT / "review"
    decisions = review / "decisions.json"
    print()
    print("Checking required inputs:")
    for label, p in [("plan",      plan),
                     ("audio",     audio),
                     ("review/",   review),
                     ("decisions", decisions)]:
        status = "✓" if p.exists() else "✗"
        print(f"  {status} {label:10s} {p}")
    missing = [p for p in (plan, audio, review, decisions) if not p.exists()]
    assert not missing, f"Missing required inputs: {missing}"
    print("\n✅ All inputs present.")

if SOURCE == "zip":
    # Upload one .zip that, when extracted in /content/output/, lands
    # plan + audio + review/ at the expected paths.  In Colab the
    # uploader is the files widget; outside Colab we expect the zip
    # already at /content/inputs.zip.
    try:
        from google.colab import files   # type: ignore
        print("Upload one .zip containing plan + audio + review/ …")
        uploaded = files.upload()
        assert uploaded, "No file uploaded"
        zip_name = next(iter(uploaded))
    except ImportError:
        zip_name = "inputs.zip"
        assert Path(zip_name).exists(), \
            "Outside Colab — put inputs.zip at /content/inputs.zip first"
    print(f"Extracting {zip_name} into {OUT} …")
    # -j would flatten; we want the zip's tree under OUT
    subprocess.run(["unzip", "-oq", zip_name, "-d", str(OUT)], check=True)
    # Common pitfall: zip wraps everything in a top-level dir like
    # "output/" — auto-flatten if the unpacked tree has only one dir
    # and it contains the expected files
    top = list(OUT.iterdir())
    if len(top) == 1 and top[0].is_dir():
        inner = top[0]
        if (inner / "review").is_dir() or any(inner.glob("*.mp3")):
            print(f"Flattening single top-level dir: {inner.name}/")
            for child in inner.iterdir():
                shutil.move(str(child), str(OUT / child.name))
            inner.rmdir()

elif SOURCE == "drive":
    from google.colab import drive   # type: ignore
    drive.mount('/content/drive', force_remount=False)
    src = Path(DRIVE_SOURCE_DIR)
    assert src.is_dir(), f"DRIVE_SOURCE_DIR not found: {src}"
    print(f"Copying from {src} → {OUT} …")
    for name in (Path(PLAN_FILE).name, Path(AUDIO_FILE).name):
        s = src / name
        d = OUT / name
        if not s.exists():
            print(f"  ⚠ {name} not found in {src}, skipping")
            continue
        shutil.copy2(s, d)
        print(f"  copied {name}")
    rev_src = src / "review"
    rev_dst = OUT / "review"
    assert rev_src.is_dir(), \
        f"Expected {rev_src} to contain decisions.json and shot folders"
    if rev_dst.exists():
        shutil.rmtree(rev_dst)
    shutil.copytree(rev_src, rev_dst)
    print(f"  copied review/ ({sum(1 for _ in rev_dst.rglob('*'))} files)")

else:
    raise ValueError(f"SOURCE must be 'zip' or 'drive', got {SOURCE!r}")

_confirm_inputs()


Upload one .zip containing plan + audio + review/ …


Saving Archive.zip to Archive.zip
Extracting Archive.zip into /content/output …

Checking required inputs:
  ✓ plan       /content/output/al_askari_plan_v2.json
  ✓ audio      /content/output/al_askari_audio.mp3
  ✓ review/    /content/output/review
  ✓ decisions  /content/output/review/decisions.json

✅ All inputs present.


In [6]:
# ════════════════════════════════════════════════════════════════════
# Render
# ════════════════════════════════════════════════════════════════════
import os
from pathlib import Path

Path(OUTPUT_FILE).parent.mkdir(parents=True, exist_ok=True)

# Backgrounded so the next cell can tail render.log. Phase 3's render
# typically takes 10-15 min at 1080p; tailing is easier than waiting on
# a frozen cell.
!python render_plan.py \
    --plan              {PLAN_FILE} \
    --audio             {AUDIO_FILE} \
    --review-dir        {REVIEW_DIR} \
    --book-cover-pick   {BOOK_COVER_PICK} \
    --book-cover-fit    {BOOK_COVER_FIT} \
    --book-cover-align  {BOOK_COVER_ALIGN} \
    --typography-family {TYPOGRAPHY_FAMILY} \
    --parallax \
    --typography-over-image \
    --no-captions \
    --grade             {GRADE} \
    --caption-backplate {CAPTION_BACKPLATE} \
    --output            {OUTPUT_FILE} \
    > output/render.log 2>&1 &

print("Rendering begins …")
print(f"  log:    output/render.log")
print(f"  output: {OUTPUT_FILE}")
print()
print("Run the next cell to tail the log until completion.")


Rendering begins …
  log:    output/render.log
  output: output/final_cut_B.mp4

Run the next cell to tail the log until completion.


In [7]:
# Monitor rendering progress
import time
from IPython.display import clear_output

log_path = "output/render.log"

print("Monitoring rendering progress...")
while True:
    try:
        # Read the log file contents
        try:
            with open(log_path, "r") as f:
                log_content = f.read()
        except FileNotFoundError:
            log_content = ""

        # Clear cell output and show the last 20 lines
        clear_output(wait=True)
        lines = log_content.splitlines()
        print("\n".join(lines[-20:]))

        # Check if the script's success signature is in the log
        if "Done in" in log_content or "Rendered video →" in log_content:
            print("\n✅ Rendering process completed successfully! Stopped monitoring.")
            break

        time.sleep(5)

    except KeyboardInterrupt:
        print("\n⚠️ Monitoring stopped manually. The script may still be running.")
        break


  [█████████████████████░░░░░░░░░]  72%  shot 81/84: typography                                                INFO  phase3.sources.decisions  Shot 82: chosen-file hit pexels_c.jpg
INFO  phase3.sources  Shot 82: review-dossier hit pexels_c.jpg
INFO  phase3.render  Shot 82: using fetched image from user_upload
INFO  phase3.render  [render 73%] shot 82/84: broll

  [██████████████████████░░░░░░░░]  73%  shot 82/84: broll                                                     INFO  phase3.render  [render 74%] shot 83/84: typography

  [██████████████████████░░░░░░░░]  74%  shot 83/84: typography                                                INFO  phase3.render  [render 75%] shot 84/84: title_card

  [██████████████████████░░░░░░░░]  75%  shot 84/84: title_card                                                INFO  phase3.render  [render 80%] concat all shots

  [████████████████████████░░░░░░]  80%  concat all shots                                                      INFO  phase3.render  [re